### Imports

In [ ]:
import sys
sys.path.insert(1, "./functions/")

import copy
import time
import math
import pandas as pd
import matplotlib.pyplot as plt
from mmdet.apis import inference_detector
import mmcv
from natsort import natsorted

from Information_Processing_Utilities import *
from Model_Processing_Utilities import *
from Filtering_Utilities import *
from Tracking_Utilities import *
from Metrics_Utilities import *

### Workspace initialization

In [ ]:
#Set the paths for differnt folders
mushroom_model_config_folder = "./configs/instance_segmentation_model/"
substrate_model_config_folder = "./configs/substrate_model/"
# reference_model_config_folder = None
#If a different model for size reference is required, 
reference_model_config_folder = "./configs/reference_model/"

working_folder = "./results/"

#Path to test images
test_set_path = "./test_set/images/"
#Path to COCO annotation json file, creates automatically tracking annotations
annotations, tracking_annotations = annotation_tracking("./test_set/annotations/test_instances.json")

#Create results folder structure
os.makedirs(working_folder,exist_ok=True)
os.makedirs(working_folder + "Predictions/",exist_ok=True)
os.makedirs(working_folder + "/Untracked/",exist_ok=True)
os.makedirs(working_folder + "/Tracked/",exist_ok=True)
os.makedirs(working_folder + "/Substrate/",exist_ok=True)

#Initialize size estimation csv file
establish_cluster_sizing(working_folder)

#Checking for available cuda/cpu
use_device = check_cuda()

#Loading instance segmentation models
mushroom_model,substrate_model,reference_model,visualizer = load_models(mushroom_model_config_folder,substrate_model_config_folder,reference_model_config_folder,use_device)

### Variable and workflow option initialization

In [ ]:
#Tracking clusters and cluster information 
post_filtering_polygons = []
post_filtering_polygons_info = []
metrics_post_filtering_polygons = []
metrics_post_filtering_polygons_info = []
pre_filtering_polygons = []
pre_filtering_polygons_info = []
post_harvest_post_filtering_polygons_info_base = []

#Baseline for tracking
baseline = []

#Tracking brightness of images for filtering
last_valid_brightness_ema = None
last_valid_brightness_residuals = []

#Saving pixel height of the substrate in images
detected_width_pixels = []
detected_height_pixels = []

#Tracking MOTA Metrics
#False Positive, False Negative, ID Switch, Ground Truth
mota_metric = [[],[],[],[]]
motaTracker = []

#Real substrate dimensions
substrate_real_width = 50
substrate_real_height = 36
# substrate_real_width = 33
# substrate_real_height = 28.5

#Select which class(es) of clusters to be cropped as separate images
#Set to -1 to crop all classes, else class numbers based on COCO annotations are from 0,1,2 etc.
labels_to_crop = [1]

#Generate images with cluster heights/widths measured in pixels
cluster_sizing_option = True
#Generate images with cropped clusters
crop_cluster_option = True
#Save images with substrate bbox
save_substrate_bbox_image = True
#Save images with reference bbox(es)
save_reference_bbox_image = True
#Save images with original prediction ids before tracking
save_untracked = True

#Confidence thresholds
confidence_score_threshold = 0.2 
overlapping_iou_threshold = 0.3
post_harvest_occluded_iou_overlap = 0.7

#Print filtering actions on detected clusters
verbose = True

### Instance segmentation growth monitoring pipeline for oyster mushroom clusters

In [ ]:
start_time = time.time()
test_set = natsorted(os.listdir(test_set_path))
test_set = [x for x in test_set if x.endswith("JPG")]
for img_num,test_img in enumerate(test_set):

    skip_image, last_valid_brightness_ema, last_valid_brightness_residuals = brightness_filter(test_img,
        path_to_image=test_set_path + test_img,
        last_valid_ema=last_valid_brightness_ema,
        last_valid_residuals=last_valid_brightness_residuals,
    )

    if skip_image:
        post_filtering_polygons.append([[0]])
        post_filtering_polygons_info.append([[0]])
        pre_filtering_polygons.append([[0]])
        pre_filtering_polygons_info.append([[0]])
        continue
    
    #Load the image
    img = mmcv.imread(test_set_path + test_img)
    #Color correction of the images for visualization
    image_for_visualization = mmcv.image.bgr2rgb(img)
    ##Process substrate, calculate size and save results
    substrate_result,averaged_height_pixels,averaged_width_pixels = process_substrate(substrate_model,
                                                                                      reference_model,
                                                                                      img,
                                                                                      image_for_visualization,
                                                                                      save_substrate_bbox_image,
                                                                                      save_reference_bbox_image,
                                                                                      working_folder,
                                                                                      test_img,
                                                                                      detected_width_pixels,
                                                                                      detected_height_pixels)
    
    #Mushroom segmentation inference
    image_result = inference_detector(mushroom_model, img)
    
    #Visualize predictions before filtering
    visualizer.add_datasample(
        "result",
        image_for_visualization,
        data_sample=image_result,
        draw_gt = None,
        wait_time=0,
        out_file=working_folder + "Predictions/before_filtering_predictions_" + test_img,
        pred_score_thr=0.01
    )

    #Keep predictions information before filtering
    before_filtering_results, before_filtering_results_info = process_results(image_result.cpu().numpy(),
                                                                              averaged_width_pixels,
                                                                              averaged_height_pixels,
                                                                              substrate_real_width,
                                                                              substrate_real_height)
    
    pre_filtering_polygons.append(before_filtering_results)
    pre_filtering_polygons_info.append(before_filtering_results_info)
    
    #Apply filters
    image_result = delete_low_confidence_predictions(image_result,confidence_score_threshold)
    # If no confident predictions are found, add placeholders for metrics and move to next frame
    if image_result.pred_instances.masks.size == 0:
        print("Finished processing image #{}: {}".format(img_num + 1, test_img))
        print("No confident predictions were found.")
        print("--------------------------------------------")
        metrics_post_filtering_polygons.append([])
        metrics_post_filtering_polygons_info.append([])
        continue
    
    image_result = delete_overlapping_with_lower_confidence(image_result,overlapping_iou_threshold)
    image_result = delete_post_background_clusters(img_num,image_result,
                                                   substrate_result,
                                                   post_harvest_post_filtering_polygons_info_base,
                                                   post_harvest_occluded_iou_overlap,
                                                   verbose)


#-----------------------------------------------------------------------------------------------------------------
    #Processing of cluster data to be used from tracking algorithm data structures
    results, results_info = process_results(image_result,
                                            averaged_width_pixels,
                                            averaged_height_pixels,
                                            substrate_real_width,
                                            substrate_real_height)     

    #Keep the hull information for all clusters
    post_filtering_polygons.append(results)
    post_filtering_polygons_info.append(results_info)

    #Show image with numbered clusters before tracking
    if save_untracked:
        save_untracked_image(image_for_visualization,
                            post_filtering_polygons,
                            working_folder,
                            test_img)

    #Tracking of predicted clusters
    post_filtering_polygons,post_filtering_polygons_info,baseline,image_result = cluster_sort(post_filtering_polygons,
                                                                                               post_filtering_polygons_info,
                                                                                               baseline,
                                                                                               image_result)

    metrics_post_filtering_polygons.append(post_filtering_polygons[-1])
    metrics_post_filtering_polygons_info.append(post_filtering_polygons_info[-1])
    
    if not post_filtering_polygons_info[-1]:
        post_filtering_polygons.pop()
        post_filtering_polygons_info.pop()
        
        if args.verbose:
            print("Finished processing image #{}: {}".format(img_num + 1, test_img))
            print("No valid predictions were found.")
            print("--------------------------------------------")
        continue
    
    #Visualize predictions after filtering
    visualizer.add_datasample(
        "result",
        image_for_visualization,
        data_sample=image_result,
        draw_gt = None,
        wait_time=0,
        out_file=working_folder + "Predictions/after_filtering_predictions_" + test_img,
        pred_score_thr=confidence_score_threshold
    )
    
    
    #Post tracking MOTA metric calculation
    mota_metric, motaTracker = get_tracking_metrics(tracking_annotations[img_num],
                                                    post_filtering_polygons[-1],
                                                    mota_metric,
                                                    motaTracker)
        
    #Visualize and crop tracked clusters 
    cluster_sizing_and_visualization(image_for_visualization,
                                     post_filtering_polygons,
                                     post_filtering_polygons_info,
                                     labels_to_crop,
                                     cluster_sizing_option,
                                     crop_cluster_option,
                                     working_folder,
                                     test_img,
                                     img_num,
                                     averaged_width_pixels,
                                     averaged_height_pixels,
                                     substrate_real_width,
                                     substrate_real_height)


    #Equalizing polygon list
    post_filtering_polygons, post_filtering_polygons_info = equalize_polygons(post_filtering_polygons,post_filtering_polygons_info)

    #Creating post-processing bbox baseline 
    if not post_harvest_post_filtering_polygons_info_base:
        post_harvest_post_filtering_polygons_info_base = copy.deepcopy(post_filtering_polygons_info[-1])
    else:
        for i in range(len(post_filtering_polygons_info[-1])):
            if post_filtering_polygons_info[-1][i]==[0]:
                continue
            if i<len(post_harvest_post_filtering_polygons_info_base):
                post_harvest_post_filtering_polygons_info_base[i] = copy.deepcopy(post_filtering_polygons_info[-1][i])
            else:
                post_harvest_post_filtering_polygons_info_base.append(copy.deepcopy(post_filtering_polygons_info[-1][i]))    
    
    if verbose:
        print("Finished processing image #{}: {}".format(img_num+1,test_img))
        print("--------------------------------------------")
    
end_time = time.time()
print("Total timelapse processing time: ", end_time - start_time)
print("Approximate processing time per image: ", (end_time - start_time)/len(test_set))

### Evaluation metrics calculation

In [ ]:
#Calculate metrics for test timelapse
pre_multiclass_TP_FP_FN = calculate_multiclass_TP_FP_FN(annotations,pre_filtering_polygons,pre_filtering_polygons_info)
pre_singleclass_TP_FP_FN = calculate_singleclass_TP_FP_FN(annotations,pre_filtering_polygons,pre_filtering_polygons_info)
post_multiclass_TP_FP_FN = calculate_multiclass_TP_FP_FN(annotations,post_filtering_polygons,post_filtering_polygons_info)
post_singleclass_TP_FP_FN = calculate_singleclass_TP_FP_FN(annotations,post_filtering_polygons,post_filtering_polygons_info)

print("Multi-class pre-filtering metrics:")
compute_mAP_mAR(pre_multiclass_TP_FP_FN)
print("----------------------")
print("Multi-class post-filtering metrics:")
compute_mAP_mAR(post_multiclass_TP_FP_FN)
print("----------------------")
print("Single-class pre-filtering metrics:")
compute_mAP_mAR(pre_singleclass_TP_FP_FN)
print("----------------------")
print("Single-class post-filtering metrics:")
compute_mAP_mAR(post_singleclass_TP_FP_FN)
print("----------------------")
print("MOTA metric:")
print(compute_mota(mota_metric))

### Growth curves visualization

In [ ]:
growth_data = pd.read_csv("./results/Cluster_Sizing.csv")
growth_data

In [ ]:
##Plot growth curves graph for Relative Cluster Area grouped by Cluster Tracking ID
#Change the time_interval value accordingly to match the image capture interval
time_interval = 2

plt.figure(figsize=(10,6))
num_clusters = int(max(growth_data["Cluster Tracking ID"]) + 1)
# Use "tab20" colormap for distinct colors
colors = cm.get_cmap("tab20", num_clusters)

for cluster_id, group in growth_data.groupby("Cluster Tracking ID"):
    plt.plot(time_interval * group["Image #"], group["Relative Cluster Area"], label=f"Cluster {cluster_id}", color=colors(cluster_id))

plt.ylabel("Relative Cluster Area (cm\u00b2)",fontsize=14,weight="bold")
# plt.ylabel("Relative Cluster Height (cm)",fontsize=14,weight="bold")
plt.xlabel("Time (hours)",fontsize=14,weight="bold")
plt.legend(title="Cluster ID",title_fontsize=14,fontsize=9)Cluster
plt.savefig("./results/clusters_relative_area.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
#Update the class mapping for the legend accordingly
label_mapping = {0: "Immature", 1: "Well-defined", 2: "Overstayed"}
label_colors = {0: "green", 1: "red", 2: "blue"}
#Determine the number of unique clusters
unique_clusters = growth_data["Cluster Tracking ID"].unique()
num_clusters = len(unique_clusters)

#Set up the figure and subplots
fig, axes = plt.subplots(nrows=int(math.ceil(num_clusters / 3)), ncols=3, figsize=(18, num_clusters * 2))
axes = axes.flatten()

#Create scatter plots for each cluster value in subplots
for i, cluster_id in enumerate(unique_clusters):
    cluster_df = growth_data[growth_data["Cluster Tracking ID"] == cluster_id]
    
    ax = axes[i]
    #Plot each scatter point based on class
    for label_value in cluster_df["Class"].unique():
        label_group = cluster_df[cluster_df["Class"] == label_value]
        ax.scatter(label_group["Image #"], label_group["Relative Cluster Area"], 
                   label=label_mapping.get(label_value), color=label_colors.get(label_value))
    
    ax.set_xlabel("Time (hours)",fontsize=14,weight="bold")
    ax.set_ylabel("Relative Cluster Area (cm\u00b2)",fontsize=14,weight="bold")
    ax.legend(title="Class",title_fontsize=16, fontsize=12,loc="lower right")

#Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig("./results/individual_cluster_relative_area.png",dpi=300)
plt.show()